# 09 - Statistics, figures and tables for the paper
Per-instance paired bootstrap 95% CIs, Wilcoxon signed-rank tests across seeds with Holm
correction, the teaser figure, and LaTeX tables written to `paper/tables/`. Every number in the
paper is produced here or in notebooks 01-08; `paper/claims_map.md` lists which.

In [ ]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]
MAIN_ENTRIES = ["fixed_T", "oracle_stable", "max_softmax", "entropy", "stability", "kl", "hidden_change", "max_conf_fan",
                "trm_post", "act", "ponder", "trm_joint", "v1gate", "done_post", "done_prop_post", "done_joint", "done_prop_joint"]

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

In [ ]:
from glwm2.eval.stats import bootstrap_ci, wilcoxon_seeds, holm
def fmt(m, lo, hi, pct=True):
    s = 100 if pct else 1
    return f"{m*s:.1f} \\tiny{{[{lo*s:.1f}, {hi*s:.1f}]}}"
tables = {}
for t in TASKS:
    r = rows[t]
    lines = [r"\begin{tabular}{l" + "rr" * len(SPLITS) + "}", r"\toprule",
             " & " + " & ".join([rf"\multicolumn{{2}}{{c}}{{{s.replace('_', '-')}}}" for s in SPLITS]) + r" \\",
             " & " + " & ".join(["Acc. (\\%)", "Steps"] * len(SPLITS)) + r" \\", r"\midrule"]
    fam_prev = None
    for e in MAIN_ENTRIES:
        if e not in set(r.entry): continue
        fam = ENTRIES[e][2]
        if fam_prev is not None and fam != fam_prev: lines.append(r"\midrule")
        fam_prev = fam
        cells = []
        for s in SPLITS:
            v = r[(r.entry == e) & (r.split == s)]
            if v.empty: cells += ["--", "--"]; continue
            m, lo, hi = bootstrap_ci(v.acc.values, n_boot=2000)
            cells += [fmt(m, lo, hi), f"{v.mean_steps.mean():.1f}"]
        lines.append(ENTRIES[e][3] + " & " + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    tables[t] = "\n".join(lines)
    p = PAPER_DIR / "tables" / f"main_{t}.tex"; p.write_text(tables[t]); WRITTEN.append(p)
print(tables["addition"][:1500])

In [ ]:
# paired tests: each completion-head variant vs each baseline, OOD-depth accuracy across seeds, Holm-corrected
tests = {}
for t in TASKS:
    r = rows[t][rows[t].split == "ood_depth"]
    pv = {}
    for o in [e for e in ["done_post", "done_prop_post", "done_joint", "done_prop_joint"] if e in set(r.entry)]:
        for b in [e for e in MAIN_ENTRIES if ENTRIES[e][2] in ("posthoc", "joint") and e in set(r.entry)]:
            A = r[r.entry == o].set_index("seed").acc
            B = r[r.entry == b].set_index("seed").acc
            common = A.index.intersection(B.index)
            if len(common) >= 2:
                pv[f"{o} vs {b}"] = wilcoxon_seeds(A[common].values, B[common].values)
    tests[t] = holm(pv)
    df_t = pd.DataFrame(tests[t]).T.sort_values("p")
    print(t, f"{int(df_t.reject.sum())}/{len(df_t)} comparisons significant after Holm (alpha 0.05)")
    display(df_t.head(12))

In [ ]:
# premature-stop table (OOD-depth and OOD-both), with per-instance bootstrap CIs pooled over seeds
lines = [r"\begin{tabular}{l" + "rr" * len(TASKS) + "}", r"\toprule",
         " & " + " & ".join([rf"\multicolumn{{2}}{{c}}{{{t}}}" for t in TASKS]) + r" \\",
         " & " + " & ".join(["OOD-depth", "OOD-both"] * len(TASKS)) + r" \\", r"\midrule"]
inst = {(t, s): entry_instances(cfgs[t], s) for t in TASKS for s in ["ood_depth", "ood_both"]}
for e in MAIN_ENTRIES:
    if e in ("oracle_stable",): continue
    cells = []
    for t in TASKS:
        for s in ["ood_depth", "ood_both"]:
            d = inst[(t, s)]
            if e not in set(d.entry): cells.append("--"); continue
            m, lo, hi = premature_ci(d, e)
            cells.append(fmt(m, lo, hi))
    lines.append(ENTRIES[e][3] + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
p = PAPER_DIR / "tables" / "premature.tex"; p.write_text("\n".join(lines)); WRITTEN.append(p)

In [ ]:
# Fig. 1 (teaser): one deep addition instance; where max-softmax halts vs where the propagating completion head halts
from glwm2.train.trainer import load_model
from glwm2.data.splits import load_split
from glwm2.eval.trajectory import per_position_rho
from glwm2.eval.results import per_instance
c = next(c for c in finished(cfgs["addition"]) if c.method == "base" and c.seed == 0)
pi = per_instance(c)["per_instance"]["ood_depth"]
data = load_split("addition", "ood_depth")
cand = np.where((pi["max_softmax"]["correct"] == 0) & (pi["done_prop_post"]["correct"] == 1))[0]
k = int(cand[np.argmax(data["depth"].numpy()[cand])]) if len(cand) else int(torch.argmax(data["depth"]))
model = load_model(c)
L = int(data["length"][k]); rho = per_position_rho("addition", {kk: v[k:k+1] for kk, v in data.items()})[0, :L+1].numpy()
T = int(data["depth"][k]) + 8
x, y = data["x"][k:k+1].cuda(), data["y"][k:k+1].cuda()
with torch.no_grad():
    pc = torch.stack([(o["logits"].argmax(-1) == y)[0, :L+1] for o in model.rollout(x, T)]).cpu().numpy()
fig, ax = plt.subplots(figsize=(3.5, 2.4))
ax.imshow(pc.T, aspect="auto", origin="lower", cmap="Greys_r", interpolation="nearest", extent=[0.5, T + 0.5, -0.5, L + 0.5], alpha=0.85)
ax.plot(rho, np.arange(L + 1), color="#D55E00", lw=1.2, label=r"carry arrives ($\rho_i$)")
ax.axvline(pi["max_softmax"]["steps"][k], color="#56B4E9", lw=1.5, ls="--", label="max-softmax halts")
ax.axvline(pi["done_prop_post"]["steps"][k], color="#009E73", lw=1.5, label="completion head halts")
ax.set_xlabel("step t"); ax.set_ylabel("output digit i"); ax.legend(fontsize=5.5, frameon=True, loc="lower right")
ax.set_title(f"{L}-digit addition, $d^*$ = {int(data['depth'][k])} (white = correct)", fontsize=7)
WRITTEN.append(savefig(fig, "fig1_teaser"))

In [ ]:
summary = {t: rows[t].groupby(["entry", "split"])[["acc", "mean_steps", "premature", "rho_depth", "rho_length_given_depth"]]
           .mean().reset_index().to_dict(orient="records") for t in TASKS}
WRITTEN.append(write_json(RESULTS_DIR / "paper_numbers.json", {"summary": summary,
               "holm": {t: tests[t] for t in TASKS}}))

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))